# Predicting shared blind spots from the models themselves (v2: pretrained-transformer loading fixed)

Follows `04_results/predict_from_models_plan.json`. v2 change: pretrained transformers are loaded exactly as published (their own final norm kept) and patch tokens are mean-pooled explicitly, instead of asking timm for average pooling (which swapped in a new, randomly initialised norm layer).

- **P1, disagreement identity:** P(both wrong) = (pA + pB - D)/2. Error rates from one half of the confirmation set; disagreement from the other half with no labels.
- **P2, pretrained similarity:** linear CKA between the backbones *before any retinal training*, on 2,000 unlabelled confirmation-set images.
- **P3, trained similarity:** CKA between the trained models' penultimate features.
- **P4, structural similarity:** shared family, architecture, pretraining data, pretraining objective.

A100, High-RAM, `HF_TOKEN` secret. About 25 minutes.

In [1]:
# Cell 1: setup
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install timm transformers
import os, glob, json, zipfile, itertools, datetime
import numpy as np, pandas as pd, torch, torch.nn as nn, timm
from scipy.stats import rankdata, spearmanr, norm, multivariate_normal
from scipy.optimize import brentq
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, CKPT, PREDS, RESULTS, FIGS = (f'{BASE}/{d}' for d in ('01_splits','02_checkpoints','03_predictions','04_results','05_figures'))
V = {  # variant: (label, family, architecture, pretraining data, objective, pretrained builder key)
 'vit_base_5class':        ('ViT sup',    'transformer', 'vit_b16', 'imagenet21k',   'supervised', 'vit_base_patch16_224'),
 'vit_base_mae_5class':    ('ViT MAE',    'transformer', 'vit_b16', 'imagenet1k',    'mae',        'vit_base_patch16_224.mae'),
 'vit_base_dinov2_5class': ('ViT DINOv2', 'transformer', 'vit_b14', 'lvd142m',       'dino',       'vit_base_patch14_dinov2.lvd142m'),
 'resnet50_5class':        ('ResNet sup', 'cnn',         'resnet50','imagenet1k',    'supervised', 'resnet50'),
 'resnet50_swsl_5class':   ('ResNet SWSL','cnn',         'resnet50','instagram1b',   'semi_weak',  'resnet50.fb_swsl_ig1b_ft_in1k'),
 'resnet50_dino_5class':   ('ResNet DINO','cnn',         'resnet50','imagenet1k',    'dino',       'dino_resnet50'),
 'densenet121_5class':     ('DenseNet',   'cnn',         'densenet121','imagenet1k', 'supervised', 'densenet121'),
 'efficientnet_b3_5class': ('EffNet',     'cnn',         'efficientnet_b3','imagenet1k','supervised','efficientnet_b3'),
 'retfound_5class':        ('RETFound',   'transformer', 'vit_l16', 'retina',        'mae',        'retfound')}
VARS = list(V); SEEDS = [0, 1, 2]
def tag(k): return '' if k == 0 else f'_seed{k}'
RETFOUND_REPO = 'iszt/RETFound_mae_meh'; DINO_RN50_URL = 'https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)

Mounted at /content/drive
device cuda


In [2]:
# Cell 2: observed targets from saved predictions (no images needed)
fr = pd.read_csv(f'{SPLITS}/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
IDS = fr.image_id.to_numpy(); G = fr[next(c for c in ('level','grade') if c in fr.columns)].to_numpy().astype(int); n = len(IDS)
P = {}
for v in VARS:
    for k in SEEDS:
        df = pd.read_csv(f'{PREDS}/eyepacs_predictions_fresh_{v}{tag(k)}_v2.csv', dtype={'image_id': str})
        pc = next(c for c in ('predicted_probability','probability','prob') if c in df.columns)
        s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(IDS); assert s.notna().all(); P[(v, k)] = s.to_numpy()
def refer(s, f):
    m = np.zeros(len(s), bool); m[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return m
F = 0.25; REF = {key: refer(s, F) for key, s in P.items()}; referable = G >= 2
def tetrachoric(p11, pa, pb):
    if min(pa, pb) <= 0 or max(pa, pb) >= 1: return np.nan
    za, zb = norm.ppf(pa), norm.ppf(pb)
    f = lambda r: multivariate_normal.cdf([za, zb], mean=[0, 0], cov=[[1, r], [r, 1]]) - p11
    try: return brentq(f, -0.999, 0.999)
    except ValueError: return np.nan
def pair_targets(a, b):
    cs = [((a, i), (b, j)) for i in SEEDS for j in SEEDS if (a != b or i < j)]
    out = {'tetra_miss': [], 'comiss_ref': [], 'comiss_g3': [], 'phi_err': []}
    for ka, kb in cs:
        ma, mb = referable & ~REF[ka], referable & ~REF[kb]; R = referable.sum()
        pa, pb, p11 = ma.sum() / R, mb.sum() / R, (ma & mb).sum() / R
        out['tetra_miss'].append(tetrachoric(p11, pa, pb)); out['comiss_ref'].append(p11 / (pa * pb))
        g3 = G >= 3; xa, xb = g3 & ~REF[ka], g3 & ~REF[kb]; T = g3.sum()
        out['comiss_g3'].append(((xa & xb).sum() / T) / ((xa.sum() / T) * (xb.sum() / T)) if xa.sum() and xb.sum() else np.nan)
        ea, eb = (referable != REF[ka]).astype(float), (referable != REF[kb]).astype(float); out['phi_err'].append(np.corrcoef(ea, eb)[0, 1])
    return {k: float(np.nanmean(v)) for k, v in out.items()}
PAIRS = [(a, b) for i, a in enumerate(VARS) for b in VARS[i:]]
T = {p: pair_targets(*p) for p in PAIRS}
QUAL = {v: float(np.mean([(referable & ~REF[(v, k)]).sum() for k in SEEDS])) for v in VARS}
print('targets computed for', len(T), 'pairs')

targets computed for 45 pairs


In [3]:
# Cell 3: P1, the disagreement identity (labels on one half, no labels on the other)
rng = np.random.default_rng(20260926); pats = fr.patient_id.astype(str).to_numpy(); up = np.unique(pats)
half = np.isin(pats, rng.choice(up, len(up) // 2, replace=False)); H1, H2 = half, ~half
P1 = {}
for a, b in PAIRS:
    if a == b: continue
    pred, act = [], []
    for i in SEEDS:
        for j in SEEDS:
            ra, rb = REF[(a, i)], REF[(b, j)]
            ea1 = (referable != ra)[H1].mean(); eb1 = (referable != rb)[H1].mean()      # error rates: labelled half
            D2 = (ra != rb)[H2].mean()                                                  # disagreement: unlabelled half
            pred.append((ea1 + eb1 - D2) / 2); act.append(((referable != ra) & (referable != rb))[H2].mean())
    P1[(a, b)] = {'pred_joint_err': float(np.mean(pred)), 'actual_joint_err': float(np.mean(act))}
e = np.array([[v['pred_joint_err'], v['actual_joint_err']] for v in P1.values()])
P1_SUMMARY = {'mean_abs_error': round(float(np.mean(np.abs(e[:, 0] - e[:, 1]))), 5), 'mean_actual': round(float(e[:, 1].mean()), 4),
              'mean_relative_error': round(float(np.mean(np.abs(e[:, 0] - e[:, 1]) / e[:, 1])), 4), 'spearman_pred_vs_actual': round(float(spearmanr(e[:, 0], e[:, 1])[0]), 4)}
print('P1 identity:', P1_SUMMARY)

P1 identity: {'mean_abs_error': 0.00229, 'mean_actual': 0.0802, 'mean_relative_error': 0.0289, 'spearman_pred_vs_actual': 0.9439}


In [4]:
# Cell 4: images for representation similarity (2,000 unlabelled confirmation-set eyes)
from torchvision import transforms
from PIL import Image
SUB = np.sort(np.random.default_rng(7).choice(IDS, 2000, replace=False)); want = set(SUB)
os.makedirs('/content/sub', exist_ok=True)
for z in sorted(glob.glob(f'{BASE}/06_fresh_images/eyepacs_fresh_batch*_v2.zip')):
    with zipfile.ZipFile(z) as zf:
        for m in zf.namelist():
            stem = os.path.splitext(os.path.basename(m))[0]
            if stem in want and not os.path.exists(f'/content/sub/{os.path.basename(m)}'):
                with open(f'/content/sub/{os.path.basename(m)}', 'wb') as fh: fh.write(zf.read(m))
SIDX = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob('/content/sub/*')}
assert len(SIDX) == len(SUB), (len(SIDX), len(SUB))
TF = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
BATCHES = [torch.stack([TF(Image.open(SIDX[i]).convert('RGB')) for i in SUB[s:s + 100]]) for s in range(0, len(SUB), 100)]
print(len(SUB), 'images loaded')

2000 images loaded


In [5]:
# Cell 5: features, pretrained and trained
from transformers import AutoModel, AutoConfig
class RF(nn.Module):
    def __init__(self, nc=5, pretrained=True):
        super().__init__(); cfg = AutoConfig.from_pretrained(RETFOUND_REPO)
        self.base = AutoModel.from_pretrained(RETFOUND_REPO, add_pooling_layer=False) if pretrained else AutoModel.from_config(cfg, add_pooling_layer=False)
        self.fc_norm = nn.LayerNorm(cfg.hidden_size, eps=1e-6); self.classifier = nn.Linear(cfg.hidden_size, nc)
    def feats(self, x, trained=True):
        h = self.base(pixel_values=x).last_hidden_state[:, 1:].mean(1); return self.fc_norm(h) if trained else h
@torch.no_grad()
def run(fn):
    return torch.cat([fn(b.to(dev)).float().cpu() for b in BATCHES]).numpy().astype(np.float64)
def pretrained_feats(v):
    key = V[v][5]
    if key == 'retfound': m = RF(pretrained=True).to(dev).eval(); f = run(lambda x: m.feats(x, trained=False))
    elif key == 'dino_resnet50':
        m = timm.create_model('resnet50', pretrained=False, num_classes=0); m.load_state_dict(torch.hub.load_state_dict_from_url(DINO_RN50_URL, map_location='cpu'), strict=False)
        m = m.to(dev).eval(); f = run(m)
    elif V[v][1] == 'transformer':
        m = timm.create_model(key, pretrained=True, num_classes=0, img_size=224).to(dev).eval()
        npt = m.num_prefix_tokens
        f = run(lambda x: m.forward_features(x)[:, npt:].mean(1))   # published final norm kept; mean of patch tokens
    else: m = timm.create_model(key, pretrained=True, num_classes=0).to(dev).eval(); f = run(m)
    del m; torch.cuda.empty_cache(); return f
def trained_feats(v):
    blob = torch.load(f'{CKPT}/eyepacs_{v}_v2.pt', map_location='cpu', weights_only=False); cfg = blob['config']
    if v == 'retfound_5class':
        m = RF(pretrained=False); m.load_state_dict(blob['state_dict'], strict=True); m = m.to(dev).eval(); f = run(lambda x: m.feats(x, True))
    else:
        kw = {'img_size': 224} if V[v][1] == 'transformer' else {}
        m = timm.create_model(cfg['arch'], pretrained=False, num_classes=5, **kw); m.load_state_dict(blob['state_dict'], strict=True); m = m.to(dev).eval()
        f = run(lambda x: m.forward_head(m.forward_features(x), pre_logits=True))
    del m; torch.cuda.empty_cache(); return f
FP, FT = {}, {}
for v in VARS:
    FP[v] = pretrained_feats(v); FT[v] = trained_feats(v); print(f'{V[v][0]:12s} pretrained {FP[v].shape}  trained {FT[v].shape}')

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

ViT sup      pretrained (2000, 768)  trained (2000, 768)


model.safetensors: reconstructing file:   0%|          |  0.00B /  343MB            

model.safetensors: downloading bytes:           |  0.00B            

...

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

RETFound     pretrained (2000, 1024)  trained (2000, 1024)


In [6]:
# Cell 6: CKA, structural similarity, and prediction of the observed targets
def cka(X, Y):
    X = X - X.mean(0); Y = Y - Y.mean(0)
    K, L = X @ X.T, Y @ Y.T
    return float((K * L).sum() / (np.linalg.norm(K) * np.linalg.norm(L)))
rows = []
for a, b in PAIRS:
    if a == b: continue
    rows.append({'a': V[a][0], 'b': V[b][0], **T[(a, b)],
                 'cka_pretrained': cka(FP[a], FP[b]), 'cka_trained': cka(FT[a], FT[b]),
                 'struct_shared': int(V[a][1] == V[b][1]) + int(V[a][2] == V[b][2]) + int(V[a][3] == V[b][3]) + int(V[a][4] == V[b][4]),
                 'mean_missed': (QUAL[a] + QUAL[b]) / 2, 'va': a, 'vb': b})
D = pd.DataFrame(rows); D.to_csv(f'{RESULTS}/predict_from_models_pairs.csv', index=False)
PRED = ['cka_pretrained', 'cka_trained', 'struct_shared']; TARG = ['tetra_miss', 'comiss_ref', 'comiss_g3', 'phi_err']
def rho_table(df):
    return {p: {t: float(spearmanr(df[p], df[t], nan_policy='omit')[0]) for t in TARG} for p in PRED}
obs = rho_table(D); rng = np.random.default_rng(11); boots = []
for _ in range(1000):
    pick = rng.choice(VARS, len(VARS), replace=True)
    sub = pd.concat([D[(D.va == x) & (D.vb == y)] for i, x in enumerate(pick) for y in pick[i + 1:] if x != y] or [D.iloc[:0]])
    if len(sub) >= 6: boots.append(rho_table(sub))
SPEAR = {p: {t: {'rho': round(obs[p][t], 3), 'ci95': [round(float(np.nanpercentile([b[p][t] for b in boots], 2.5)), 3), round(float(np.nanpercentile([b[p][t] for b in boots], 97.5)), 3)]}
             for t in TARG} for p in PRED}
print(pd.DataFrame({p: {t: f"{SPEAR[p][t]['rho']:+.2f} {SPEAR[p][t]['ci95']}" for t in TARG} for p in PRED}).to_string())

/tmp/ipykernel_10115/2441129200.py:16: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return {p: {t: float(spearmanr(df[p], df[t], nan_policy='omit')[0]) for t in TARG} for p in PRED}


                   cka_pretrained            cka_trained          struct_shared
tetra_miss  +0.46 [-0.644, 0.941]   +0.55 [-0.342, 0.94]  +0.51 [-0.277, 0.929]
comiss_ref  +0.40 [-0.326, 0.932]  +0.45 [-0.396, 0.953]  -0.15 [-0.771, 0.538]
comiss_g3   +0.17 [-0.604, 0.867]  -0.14 [-0.759, 0.765]   -0.27 [-0.816, 0.55]
phi_err     +0.28 [-0.666, 0.932]  +0.29 [-0.472, 0.909]   +0.65 [-0.102, 0.94]


In [7]:
# Cell 7: variance components, figure, write
VC = []
for a, b in PAIRS:
    VC.append({'tetra': T[(a, b)]['tetra_miss'], 'seed_sib': int(a == b), 'same_arch': int(V[a][2] == V[b][2]), 'same_family': int(V[a][1] == V[b][1]),
               'same_data': int(V[a][3] == V[b][3]), 'same_obj': int(V[a][4] == V[b][4])})
VC = pd.DataFrame(VC).dropna(); X = np.column_stack([np.ones(len(VC)), VC[['same_family','same_arch','same_data','same_obj','seed_sib']].values])
beta, *_ = np.linalg.lstsq(X, VC.tetra.values, rcond=None)
VCOMP = dict(zip(['floor_intercept','same_family','same_architecture','same_pretraining_data','same_objective','seed_sibling_extra'], np.round(beta, 4).tolist()))
print('variance components (latent correlation of misses):', VCOMP)
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
col = {0: '#999999', 1: '#56B4E9', 2: '#009E73', 3: '#E69F00', 4: '#D55E00'}
for ax, p, lab in zip(axes, PRED, ('Pretrained similarity (CKA, before retinal training)', 'Trained similarity (CKA)', 'Shared structural attributes')):
    ax.scatter(D[p], D.tetra_miss, c=[col[s] for s in D.struct_shared], s=55, edgecolor='#333', lw=0.4)
    ax.set_xlabel(lab); ax.set_ylabel('Latent correlation of referable misses' if p == PRED[0] else '')
    ax.set_title(f"rho = {SPEAR[p]['tetra_miss']['rho']:+.2f} {SPEAR[p]['tetra_miss']['ci95']}", loc='left', fontsize=9.5, weight='bold')
    ax.spines[['top','right']].set_visible(False)
fig.suptitle('Can the models themselves predict their shared blind spots?', x=0.01, ha='left', fontsize=12, weight='bold'); fig.tight_layout()
fig.savefig(f'{FIGS}/predict_from_models_v2.png', dpi=250, bbox_inches='tight'); plt.close(fig)
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'predict_from_models_plan.json', 'status': 'exploratory',
       'P1_disagreement_identity': P1_SUMMARY, 'spearman_across_36_pairs': SPEAR, 'variance_components': VCOMP,
       'signal_pretrained_cka_predicts': bool(SPEAR['cka_pretrained']['tetra_miss']['ci95'][0] > 0), 'n_images_for_cka': int(len(SUB))}
with open(f'{RESULTS}/predict_from_models_results.json', 'w') as f: json.dump(res, f, indent=2)
print('signal (pretrained CKA predicts shared misses):', res['signal_pretrained_cka_predicts'])
print('Wrote 04_results/predict_from_models_results.json, predict_from_models_pairs.csv, 05_figures/predict_from_models_v2.png')

variance components (latent correlation of misses): {'floor_intercept': 0.764, 'same_family': 0.0111, 'same_architecture': 0.0156, 'same_pretraining_data': 0.0146, 'same_objective': 0.0041, 'seed_sibling_extra': 0.0956}
signal (pretrained CKA predicts shared misses): False
Wrote 04_results/predict_from_models_results.json, predict_from_models_pairs.csv, 05_figures/predict_from_models_v2.png
